# 02 – Data quality check
Before any analysis, check what is missing, odd or out of scope in the combined dataset.
This notebook only **looks** at the data; the fixes are applied in `03_clean.ipynb`.

In [10]:
import pandas as pd

df = pd.read_csv("../data/processed/complaints_combined.csv", low_memory=False)
print(f"{len(df):,} rows, {df.shape[1]} columns")
df.head()

569,138 rows, 16 columns


,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,source_file
0,2023-03-11T14:52:49.000Z,Checking or savings account,Savings account,Problem with a lender or other company chargin...,Transaction was not authorized,Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11367,Older American,Web,2023-03-11T15:09:00.000Z,Closed with explanation,Yes,6681481,checking_2023.csv
1,2023-03-11T15:26:03.000Z,Checking or savings account,Checking account,Problem with a lender or other company chargin...,Money was taken from your account on the wrong...,Company has responded to the consumer and the ...,TRUIST FINANCIAL CORPORATION,GA,30097,Servicemember,Web,2023-03-11T15:41:06.000Z,Closed with explanation,Yes,6681394,checking_2023.csv
2,2023-03-11T15:39:48.000Z,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,Company has responded to the consumer and the ...,"BANK OF AMERICA, NATIONAL ASSOCIATION",FL,33618,NaN,Web,2023-03-11T16:21:40.000Z,Closed with explanation,Yes,6681219,checking_2023.csv
3,2023-03-11T15:40:02.000Z,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,Company has responded to the consumer and the ...,WELLS FARGO & COMPANY,SC,29657,NaN,Web,2023-03-11T15:54:03.000Z,Closed with explanation,Yes,6681452,checking_2023.csv
4,2023-03-11T15:53:46.000Z,Checking or savings account,Checking account,Problem with a lender or other company chargin...,Money was taken from your account on the wrong...,Company has responded to the consumer and the ...,BMO BANK NATIONAL ASSOCIATION,IL,60623,NaN,Web,2023-03-11T16:00:09.000Z,Closed with explanation,Yes,6681372,checking_2023.csv


## Check 1 – Missing values

In [11]:
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "pct": (df.isna().mean() * 100).round(2)
})
missing[missing["missing"] > 0].sort_values("missing", ascending=False)

,missing,pct
Tags,478095,84.00
Company public response,299096,52.55
State,3872,0.68
ZIP code,984,0.17
Sub-issue,451,0.08
Issue,5,0.00
Sub-product,5,0.00


## Check 2 – Dates
Dates come with a time and a UTC marker (e.g. `2024-01-01T01:53:02.000Z`). Convert them and check the gap between the CFPB receiving a complaint and sending it to the company.

In [12]:
df["date_received"] = pd.to_datetime(df["Date received"], utc=True)
df["date_sent"] = pd.to_datetime(df["Date sent to company"], utc=True)

print("Received from", df["date_received"].min().date(), "to", df["date_received"].max().date())

lag = (df["date_sent"] - df["date_received"]).dt.days
print("\nDays from received to sent to company:")
print(lag.describe().round(1))
print("\nSent BEFORE received (impossible):", (lag < 0).sum())
print("Took more than 30 days:", (lag > 30).sum())

Received from 2023-01-01 to 2026-11-14

Days from received to sent to company:
count    569138.0
mean          1.2
std           7.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max         445.0
dtype: float64

Sent BEFORE received (impossible): 0
Took more than 30 days: 8643


## Check 3 – Company response outcomes
'In progress' complaints have no final outcome yet, so they can't count towards relief rates.

In [13]:
print(df["Company response to consumer"].value_counts(), "\n")
print(df["Timely response?"].value_counts(), "\n")

in_progress = df[df["Company response to consumer"] == "In progress"]
print("In progress complaints by month received:")
print(in_progress["date_received"].dt.strftime("%Y-%m").value_counts().sort_index().tail(6))

Company response to consumer
Closed with explanation            411214
Closed with monetary relief         76694
Closed with non-monetary relief     71949
In progress                          8654
Untimely response                     627
Name: count, dtype: int64 

Timely response?
Yes    565487
No       3651
Name: count, dtype: int64 

In progress complaints by month received:
date_received
2026-06       1
2026-07      12
2026-08      92
2026-09    1236
2026-10    6292
2026-11    1017
Name: count, dtype: int64


## Check 4 – Companies
This project is about **banks**. Check whether non-bank companies appear.

In [14]:
print("Unique companies:", df["Company"].nunique(), "\n")
print(df["Company"].value_counts().head(15))

Unique companies: 1209 

Company
JPMORGAN CHASE & CO.                      52420
CAPITAL ONE FINANCIAL CORPORATION         51461
WELLS FARGO & COMPANY                     43778
CITIBANK, N.A.                            43117
BANK OF AMERICA, NATIONAL ASSOCIATION     42090
SYNCHRONY FINANCIAL                       24721
NAVY FEDERAL CREDIT UNION                 21890
Chime Financial Inc                       20409
EQUIFAX, INC.                             20262
TRANSUNION INTERMEDIATE HOLDINGS, INC.    19311
AMERICAN EXPRESS COMPANY                  18358
Experian Information Solutions Inc.       18164
U.S. BANCORP                              12342
Bread Financial Holdings, Inc.            11863
GOLDMAN SACHS BANK USA                    11522
Name: count, dtype: int64


In [15]:
bureaus = df["Company"].str.contains("EQUIFAX|TRANSUNION|EXPERIAN", case=False)
print(f"Credit bureau complaints: {bureaus.sum():,} ({bureaus.mean():.1%} of all rows)\n")
print(df[bureaus]["Issue"].value_counts().head(5))

Credit bureau complaints: 57,737 (10.1% of all rows)

Issue
Problem with a company's investigation into an existing problem                     21744
Incorrect information on your report                                                21552
Getting a credit card                                                                7664
Problem with a credit reporting company's investigation into an existing problem     3596
Other features, terms, or problems                                                    649
Name: count, dtype: int64


## Check 5 – States
The US has 50 states + DC. Look for anything else.

In [16]:
us_50_dc = ["AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA","ME",
            "MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI",
            "SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY"]
other = df[df["State"].notna() & ~df["State"].isin(us_50_dc)]
print(f"Rows with a state outside the 50 + DC: {len(other):,}")
print(other["State"].value_counts())

Rows with a state outside the 50 + DC: 1,669
State
PR                                      1114
AE                                       208
AP                                       156
VI                                        82
GU                                        56
UNITED STATES MINOR OUTLYING ISLANDS      28
AS                                        13
MP                                         6
AA                                         5
PW                                         1
Name: count, dtype: int64


## Check 6 – Odd categories

In [17]:
print(df["Sub-product"].value_counts(dropna=False))

Sub-product
General-purpose credit card or charge card    269969
Checking account                              218434
Store credit card                              33698
Other banking product or service               21687
Savings account                                20436
CD (Certificate of Deposit)                     4908
NaN                                                5
Credit reporting                                   1
Name: count, dtype: int64


In [18]:
print(df["ZIP code"].astype(str).head(10).tolist())
print("ZIPs masked with XX:", df["ZIP code"].astype(str).str.contains("XX").sum())

['11367', '30097', '33618', '29657', '60623', '20743', '60564', '206XX', '32808', '30043']
ZIPs masked with XX: 81363


## Data Quality Summary

| Data Quality Check | Observation | Action Taken |
|---|---|---|
| Data consolidation | Data was downloaded in 9 separate files based on product and year. | Combined all files into one dataset for analysis. |
| Product consistency | The dataset included an older product category, "Credit card or prepaid card," which also contained prepaid and gift card complaints. | Removed prepaid/gift card records to keep the analysis focused on credit cards and checking/savings accounts. |
| Company relevance | Complaints against Equifax, TransUnion, and Experian were included, but these are credit reporting agencies rather than banks. | Excluded complaints against credit bureaus from the bank-focused analysis. |
| Missing or invalid categories | Some records had missing or invalid category values. | Removed 6 records that could not be reliably classified. |
| Date validity | Checked whether any complaint was sent to a company before it was received. None were found. | Dates were treated as reliable and kept as they are. |
| Complaint status | Some complaints were still marked as "In progress" and did not have a final outcome. | Included them in complaint volume analysis but excluded them from relief rate calculations. |
| Blank text fields | Most complaints have no tag or public company response. Filling them with the word "None" caused pandas to read them back as missing values. | Filled them with "No tag" and "No public response" instead. |
| State information | Some state values were blank, and some records represented military or territory locations. | Replaced blank state values with "Unknown" and grouped military and territory codes together. |
| Geographic data | ZIP codes were partially masked by the CFPB for privacy reasons. | Did not use ZIP codes in the analysis. |
| KPI preparation | Outcome fields were not directly available in the format required for analysis. | Created flags for closed complaints, relief received, monetary relief received, and timely responses. |